1. ##### Monthly Recurring Revenue (MRR) & Active Subscriptions by Plan Tier

In [0]:
%sql
SELECT 
    p.plan_name,
    p.tier,
    COUNT(f.subscription_id) AS total_subscriptions,
    SUM(CASE WHEN f.is_active THEN 1 ELSE 0 END) AS active_subscriptions,
    ROUND(SUM(f.mrr_amount), 2) AS total_mrr
FROM adb_practice.gold.fact_subscriptions f
JOIN adb_practice.gold.dim_plan p 
    ON f.plan_key = p.plan_key
GROUP BY p.plan_name, p.tier
ORDER BY total_mrr DESC;

plan_name,tier,total_subscriptions,active_subscriptions,total_mrr
Enterprise Custom,enterprise,68,65,64935.0
Enterprise Scale,enterprise,67,63,18837.0
Pro Monthly,pro,74,69,4071.0
Pro Annual,pro,73,71,3479.0
Free Tier,starter,62,55,0.0


2. ##### Cash Collection & Invoicing Anomaly Auditing

In [0]:
%sql
SELECT 
    d.year,
    d.quarter,
    COUNT(b.invoice_id) AS total_invoices,
    ROUND(SUM(b.amount_due), 2) AS total_invoiced,
    ROUND(SUM(b.amount_paid), 2) AS total_collected,
    ROUND(100.0 * SUM(b.amount_paid) / NULLIF(SUM(b.amount_due), 0), 2) AS collection_rate_pct,
    SUM(CASE WHEN b.is_anomaly THEN 1 ELSE 0 END) AS anomaly_count
FROM adb_practice.gold.fact_billing b
JOIN adb_practice.gold.dim_date d 
    ON b.date_key = d.date_key
WHERE b.date_key != '-1'
GROUP BY d.year, d.quarter
ORDER BY d.year DESC, d.quarter DESC;

year,quarter,total_invoices,total_invoiced,total_collected,collection_rate_pct,anomaly_count
2026,4,32,5508.0,4534.0,82.32,1
2026,3,244,70191.0,64211.0,91.48,3
2026,2,241,69253.0,62641.0,90.45,2
2026,1,255,72094.0,66153.0,91.76,5
2025,4,220,54362.0,48843.0,89.85,6


3. ##### Daily Active Users (DAU) & Security Failure Rate

In [0]:
%sql
SELECT 
    d.date_actual,
    COUNT(DISTINCT f.user_key) AS unique_active_users,
    COUNT(f.login_fact_key) AS total_login_attempts,
    ROUND(100.0 * SUM(CASE WHEN f.is_success THEN 1 ELSE 0 END) / COUNT(f.login_fact_key), 2) AS login_success_pct
FROM adb_practice.gold.fact_logins f
JOIN adb_practice.gold.dim_date d 
    ON f.date_key = d.date_key
WHERE d.date_key != '-1'
GROUP BY d.date_actual
ORDER BY d.date_actual DESC
LIMIT 10;

date_actual,unique_active_users,total_login_attempts,login_success_pct
2026-10-12,5,5,80.00
2026-10-11,172,176,93.18
2026-10-10,153,161,92.55
2026-10-09,164,171,95.91
2026-10-08,150,159,90.57
2026-10-07,162,174,92.53
2026-10-06,150,154,90.91
2026-10-05,171,177,93.22
2026-10-04,175,182,93.41
2026-10-03,174,184,91.30


4. ##### Top Feature Consumption by Industry

In [0]:
%sql
SELECT 
    o.industry,
    f.feature_name,
    SUM(f.usage_count) AS total_usage
FROM adb_practice.gold.fact_feature_usage f
JOIN adb_practice.gold.dim_organization o 
    ON f.org_key = o.org_key
WHERE f.org_key != '-1'
GROUP BY o.industry, f.feature_name
ORDER BY o.industry, total_usage DESC;

industry,feature_name,total_usage
E-commerce,dashboard_view,104299
E-commerce,report_export,103025
E-commerce,seat_added,99261
E-commerce,api_calls,98216
E-commerce,webhook_trigger,93989
E-commerce,unknown_feature,21237
EdTech,seat_added,90800
EdTech,dashboard_view,82976
EdTech,webhook_trigger,80421
EdTech,api_calls,74722
